# Motion-S Colab bootstrap & training

End-to-end runner for [Motion-S Hierarchical Text-to-Motion Generation for Sign Language](https://www.kaggle.com/competitions/motion-s-hierarchical-text-to-motion-generation-for-sign-language).

## 1. Kaggle authentication

Run the cell, paste your Kaggle API token when prompted (from Kaggle â†’ Account â†’ API â†’ *Create New Token*).


In [1]:
import kagglehub

kagglehub.login()
print("kagglehub login OK")


kagglehub login OK


## 2. Download competition data


In [10]:
import os
os.environ["KAGGLE_CONFIG_DIR"] = "/root/.kaggle"

import kagglehub
COMP = "motion-s-hierarchical-text-to-motion-generation-for-sign-language"
DATA_PATH = kagglehub.competition_download(COMP)
print("data:", DATA_PATH)
get_ipython().system(f"ls -lh '{DATA_PATH}'")


Resuming download from 181403648 bytes (14255055090 bytes left)...
Resuming download to /root/.cache/kagglehub/competitions/motion-s-hierarchical-text-to-motion-generation-for-sign-language.archive (181403648/14436458738) bytes left.


100%|██████████| 13.4G/13.4G [12:12<00:00, 19.5MB/s] 

Extracting files...


data: /root/.cache/kagglehub/competitions/motion-s-hierarchical-text-to-motion-generation-for-sign-language
total 32M
drwxr-xr-x     2 root root 340K May 20 10:45 Motion-Features
-rw-r--r--     1 root root  716 May 20 10:46 sample_submission.csv
-rw-r--r--     1 root root 210K May 20 10:46 test.csv
drwxr-xr-x 12470 root root 264K May 20 10:46 Train
-rw-r--r--     1 root root  31M May 20 10:46 train.csv


## 3. Clone repo, link data, install deps


In [11]:
import os, subprocess, pathlib

REPO_URL = "https://github.com/Axentalan-VI/Motion-S-Text-to-Sign-Motion-Generation-Signvrse.git"
PROJECT_DIR = "/content/motion-s"

if pathlib.Path(PROJECT_DIR, ".git").exists():
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
print("cwd:", os.getcwd())

# data symlink
data_link = pathlib.Path(PROJECT_DIR) / "data"
if data_link.is_symlink() or data_link.exists():
    if data_link.is_symlink():
        data_link.unlink()
    elif data_link.is_dir() and not any(data_link.iterdir()):
        data_link.rmdir()
data_link.symlink_to(DATA_PATH)
print("data ->", os.readlink(data_link))

# deps
get_ipython().system("pip -q install ftfy regex git+https://github.com/openai/CLIP.git")
get_ipython().system("ls data/")


cwd: /content/motion-s
data -> /root/.cache/kagglehub/competitions/motion-s-hierarchical-text-to-motion-generation-for-sign-language
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
Motion-Features  sample_submission.csv	test.csv  Train  train.csv


## 4. Sanity check (GPU + dataset)


In [12]:
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

from src.data.io import load_train
df = load_train()
print(f"train rows: {len(df)}  cols: {list(df.columns)[:6]}...")
print(f"seq_len p50={df.seq_len.median():.0f}  p95={df.seq_len.quantile(0.95):.0f}")


torch 2.10.0+cu128 cuda True NVIDIA A100-SXM4-80GB
train rows: 12467  cols: ['id', 'sentence', 'gloss', 'bvh_path', 'base_tokens', 'residual_1']...
seq_len p50=98  p95=181


## 5. Freeze split + configure Drive mirror


In [13]:
get_ipython().system("python -m scripts.make_split")

BACKUP_MODE = "download"   # "drive" | "download" | None
DRIVE_CKPTS = "/content/drive/MyDrive/motion-s-ckpts"

DRIVE_FLAG = ""
if BACKUP_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_FLAG = f" --drive-dir {DRIVE_CKPTS}"

import os
def backup_ckpt(name: str) -> None:
    """Persist a freshly-saved checkpoint by the chosen strategy."""
    path = f"checkpoints/{name}"
    if not os.path.exists(path):
        print(f"[backup] missing: {path}"); return
    if BACKUP_MODE == "download":
        from google.colab import files
        print(f"[backup] downloading {path} ...")
        files.download(path)
    elif BACKUP_MODE == "drive":
        # already mirrored by trainer via --drive-dir; nothing to do.
        print(f"[backup] mirrored via --drive-dir to {DRIVE_CKPTS}/{name}")
    else:
        print(f"[backup] skipped ({path} stays in /content only)")

print(f"BACKUP_MODE={BACKUP_MODE!r}  DRIVE_FLAG={DRIVE_FLAG!r}")


loading /content/motion-s/data/train.csv ...
  rows: 12467
train: 11220  val: 1247  -> /content/motion-s/data/split_90_10.json
BACKUP_MODE='download'  DRIVE_FLAG=''


## 6. Train length predictor (~10 min)


In [14]:
get_ipython().system(f"python -m scripts.train_length --epochs 10{DRIVE_FLAG}")
backup_ckpt("length_predictor.pth")


[load] /content/motion-s/data/train.csv
[data] 12467 rows, seq_len p50=98 p95=181
[split] train=11220  val=1247  (covered 12467/12467)
[filter] train kept 11139/11220, val kept 1234/1247
config.json: 100% 483/483 [00:00<00:00, 1.88MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 209kB/s]
vocab.txt: 232kB [00:00, 761kB/s]
tokenizer.json: 466kB [00:00, 1.88MB/s]
[model] LengthPredictor(num_bins=32, backbone=distilbert-base-uncased)
model.safetensors: 100% 268M/268M [00:02<00:00, 110MB/s] 
Loading weights: 100% 100/100 [00:00<00:00, 1295.47it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECT

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Train MoMask base (30 epochs + EMA)


In [15]:
get_ipython().system(f"python -m scripts.train_momask_base --epochs 30 --batch-size 32{DRIVE_FLAG}")
backup_ckpt("momask_base.pth")


[load] /content/motion-s/data/train.csv
[data] 12467 rows; seq_len p50=98 p95=181
[split] train=11220  val=1247
[dataset] train kept=11162  val kept=1236
100%|████████████████████████████████████████| 338M/338M [00:00<00:00, 382MiB/s]
[ema] enabled, decay=0.999
[model] BaseMaskTransformer params=34.6M  cfg=MoMaskConfig(d_model=512, n_layers=8, n_heads=8, d_ff=2048, dropout=0.1, max_len=320, vocab_size=514, cond_dim=512, n_residual_layers=5, num_length_bins=32)
  [ep00 it0050] loss=5.445 acc=0.054 lr=2.00e-04
  [ep00 it0100] loss=5.218 acc=0.086 lr=2.00e-04
  [ep00 it0150] loss=5.080 acc=0.106 lr=2.00e-04
  [ep00 it0200] loss=4.957 acc=0.124 lr=2.00e-04
  [ep00 it0250] loss=4.866 acc=0.135 lr=2.00e-04
  [ep00 it0300] loss=4.797 acc=0.143 lr=2.00e-04
[ep 00] train_loss=4.734 acc=0.151  val_loss=5.296 acc=0.080
        ↳ new best, saved -> /content/motion-s/checkpoints/momask_base.pth
  [ep01 it0050] loss=4.149 acc=0.216 lr=1.99e-04
  [ep01 it0100] loss=4.117 acc=0.221 lr=1.99e-04
  [ep01

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Train MoMask residual


In [16]:
get_ipython().system(f"python -m scripts.train_momask_residual --epochs 10 --batch-size 32{DRIVE_FLAG}")
backup_ckpt("momask_residual.pth")


[load] /content/motion-s/data/train.csv
[data] 12467 rows
[split] train=11220  val=1247
[dataset] train kept=11162  val kept=1236
[model] ResidualTransformer params=35.9M  cfg=MoMaskConfig(d_model=512, n_layers=8, n_heads=8, d_ff=2048, dropout=0.1, max_len=320, vocab_size=514, cond_dim=512, n_residual_layers=5, num_length_bins=32)
  [ep00 it0050] loss=5.690 acc=0.056 lr=2.00e-04
  [ep00 it0100] loss=5.430 acc=0.078 lr=2.00e-04
  [ep00 it0150] loss=5.291 acc=0.092 lr=1.99e-04
  [ep00 it0200] loss=5.180 acc=0.105 lr=1.98e-04
  [ep00 it0250] loss=5.064 acc=0.119 lr=1.97e-04
  [ep00 it0300] loss=4.961 acc=0.131 lr=1.96e-04
[ep 00] train_loss=4.876 acc=0.142  val_loss=4.269 acc=0.215  L1=0.27 L2=0.24 L3=0.20 L4=0.19 L5=0.18
        ↳ new best, saved -> /content/motion-s/checkpoints/momask_residual.pth
  [ep01 it0050] loss=4.236 acc=0.220 lr=1.94e-04
  [ep01 it0100] loss=4.255 acc=0.218 lr=1.92e-04
  [ep01 it0150] loss=4.218 acc=0.223 lr=1.90e-04
  [ep01 it0200] loss=4.190 acc=0.227 lr=1.88e

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Score on val (R-Precision / FID / Diversity)


In [27]:
import os, glob

candidates = sorted(
    glob.glob("data/**/Public_t2m_align.pth", recursive=True)
    + glob.glob(f"{DATA_PATH}/**/Public_t2m_align.pth", recursive=True)
)
if not candidates:
    raise FileNotFoundError(
        "Public_t2m_align.pth not found under data/ or DATA_PATH. "
        "Run the competition-files probe cell to confirm it was published."
    )
EVAL_CKPT = candidates[0]
print(f"[evaluator] using {EVAL_CKPT}  ({os.path.getsize(EVAL_CKPT)/1e6:.1f} MB)")

get_ipython().system(
    f"python -m scripts.score_momask --n 256 --batch-size 16 "
    f"--evaluator-ckpt '{EVAL_CKPT}'"
)


FileNotFoundError: Public_t2m_align.pth not found under data/ or DATA_PATH. Run the competition-files probe cell to confirm it was published.

In [19]:

import os, shutil
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DEST_DIR = "/content/drive/MyDrive/motion-s-ckpts"
os.makedirs(DEST_DIR, exist_ok=True)

for n in ["length_predictor.pth", "momask_base.pth", "momask_residual.pth"]:
    src = f"checkpoints/{n}"
    if os.path.exists(src):
        shutil.copy2(src, f"{DEST_DIR}/{n}")
        print(f"[ok]   {src} -> {DEST_DIR}/{n}  ({os.path.getsize(src)/1e6:.1f} MB)")
    else:
        print(f"[miss] {src}")
print("done.")


Mounted at /content/drive
[ok]   checkpoints/length_predictor.pth -> /content/drive/MyDrive/motion-s-ckpts/length_predictor.pth  (4.9 MB)
[ok]   checkpoints/momask_base.pth -> /content/drive/MyDrive/motion-s-ckpts/momask_base.pth  (553.9 MB)
[ok]   checkpoints/momask_residual.pth -> /content/drive/MyDrive/motion-s-ckpts/momask_residual.pth  (429.1 MB)
done.


In [23]:
import json, os
from pathlib import Path
import kagglehub

get_ipython().system("pip -q install kaggle")

# kagglehub stores creds in memory only; write them so the kaggle CLI works.
try:
    creds = kagglehub.auth.get_kaggle_credentials() 
    user = getattr(creds, "username", None) or creds[0]
    key  = getattr(creds, "key", None)      or creds[1]
    kdir = Path.home() / ".kaggle"
    kdir.mkdir(exist_ok=True)
    (kdir / "kaggle.json").write_text(json.dumps({"username": user, "key": key}))
    (kdir / "kaggle.json").chmod(0o600)
    os.environ["KAGGLE_USERNAME"] = user
    os.environ["KAGGLE_KEY"] = key
    print(f"kaggle.json written for user: {user}")
except Exception as e:
    print(f"creds export failed: {e!r}")

print("\nâ”€â”€ Files published in competition â”€â”€")
get_ipython().system(f"kaggle competitions files {COMP} 2>&1 | head -200")
print("\nâ”€â”€ Filtered for evaluator/.pth â”€â”€")
get_ipython().system(f"kaggle competitions files {COMP} 2>&1 | grep -iE 'evaluator|align|\\.pth' || echo '(no matches)'")


creds export failed: TypeError("'KaggleApiCredentials' object is not subscriptable")

â”€â”€ Files published in competition â”€â”€
You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication

â”€â”€ Filtered for evaluator/.pth â”€â”€
(no matches)


In [28]:
creds = kagglehub.auth.get_kaggle_credentials() 
user = getattr(creds, "username", None) or creds[0]
key  = getattr(creds, "key", None)      or creds[1]

TypeError: 'KaggleApiCredentials' object is not subscriptable

In [25]:
!git -C /content/motion-s pull


Already up to date.
